In [1]:
pip install -r "/kaggle/input/datasets/lamaal25/requirements/requirements.txt"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.1 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install -U llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 25.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.35-py3-none-linux_x86_64.whl size=21110793 sha256=8b205e9391a0ee7be4c2ed0e83ffa5c6040bb1f45d78475418a0f07437e28467
  Stored in directory: /root/.cache/pip/wheels/1b/64/d4/17744d793e69b485a7664ef47b18e402a72a6e08e84f7b9926
Successfully built llama-cpp-python


In [3]:
import os, json, random
import numpy as np
import torch

# ─── عدّلي هذا حسب اسم الـ Kaggle Dataset اللي رفعتِ فيه ملفات JSONL ───
DATA_DIR = "/kaggle/input/datasets/lamaal25/lessonsdataset"   # غيّريه لاسم الداتاسِت عندك

TRAIN_PATH = f"{DATA_DIR}/merged_train.jsonl"
VAL_PATH   = f"{DATA_DIR}/merged_validation.jsonl"
TEST_PATH  = f"{DATA_DIR}/merged_test.jsonl"

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"   # عدّليه لو الاسم مختلف على HF

OUTPUT_DIR = "/kaggle/working/midad-allam-lora"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("GPU متاح:", torch.cuda.is_available(), "| عدد الـ GPUs:", torch.cuda.device_count())


GPU متاح: True | عدد الـ GPUs: 2


In [4]:
from datasets import Dataset

def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(TRAIN_PATH)
val_rows   = read_jsonl(VAL_PATH)
test_rows  = read_jsonl(TEST_PATH)

print(f"train={len(train_rows)}  validation={len(val_rows)}  test={len(test_rows)}")

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)
test_ds  = Dataset.from_list(test_rows)


train=186  validation=58  test=63


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("ما لقيت HF_TOKEN بالـ Secrets — تجاهلي هذا لو النموذج مو gated.")
    
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

ما لقيت HF_TOKEN بالـ Secrets — تجاهلي هذا لو النموذج مو gated.


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
def format_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds_fmt = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_ds_fmt   = val_ds.map(format_example, remove_columns=val_ds.column_names)

print(train_ds_fmt[0]["text"][:800])


Map:   0%|          | 0/186 [00:00<?, ? examples/s]

Map:   0%|          | 0/58 [00:00<?, ? examples/s]

<s> [INST] <<SYS>>
أنت كاتب قصص تعليمية للأطفال من عمر 7 إلى 10 سنوات. مهمتك تحويل الدروس التعليمية إلى قصص ممتعة وسهلة الفهم مع المحافظة على دقة المعلومات وإضافة لمسات طبيعية من البيئة والثقافة السعودية. عند ذكر مكان القصة، استخدمي دائماً اسم مدينة أو منطقة سعودية حقيقية ومميزة (مثل الأحساء، أبها، العُلا، القصيم، جدة، الطائف، حائل، نجران، الدمام) تُضفي على القصة نكهة محلية واضحة، وتجنّبي وصف المكان بكلمة عامة مجردة من دون اسم مدينة أو منطقة (مثل «المدرسة» أو «المنزل» أو «الفصل» أو «المسجد» لوحدها).
<</SYS>>

عنوان الدرس:
أنواع الحيوانات

أهداف التعلم:
* أن يصنّف الطفل الحيوانات إلى مجموعات (ثدييات، طيور، زواحف، برمائيات، أسماك، حشرات).
* أن يذكر الطفل خصائص كل مجموعة من مجموعات الحيوانات.
* أن يميّز الطفل بين الحيوانات المتشابهة والمختلفة في مجموعاتها.

المفاهيم الأساسية:
* الثدييات
* الط


In [7]:
lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds_fmt["text"]]
print("أقصى طول:", max(lengths), "| متوسط:", int(sum(lengths)/len(lengths)), "| أطول ١٠٪:", int(np.percentile(lengths, 90)))

MAX_SEQ_LENGTH = int(np.percentile(lengths, 99)) + 64  # هامش أمان بسيط
MAX_SEQ_LENGTH = max(512, min(MAX_SEQ_LENGTH, 4096))
print("MAX_SEQ_LENGTH المُختار:", MAX_SEQ_LENGTH)

أقصى طول: 2076 | متوسط: 1313 | أطول ١٠٪: 1591
MAX_SEQ_LENGTH المُختار: 2128


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 39,976,960 || all params: 7,040,536,576 || trainable%: 0.5678


In [10]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,     # حجم دفعة فعّال = 16
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=1)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
 
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
    # المعاملات الثلاثة اللي كانت تروح لـ SFTTrainer، صارت هنا:
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    loss_type="nll",
)
 
early_stopping = EarlyStoppingCallback(early_stopping_patience=1)
 
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_fmt,
    eval_dataset=val_ds_fmt,
    processing_class=tokenizer,   # لاحظي: tokenizer= صار deprecated، استخدمي processing_class= بدلها
    callbacks=[early_stopping],
)
 
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/186 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/186 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/186 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/186 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [16]:
python !nvidia-smi

SyntaxError: invalid syntax (2458808868.py, line 1)